# R-GCN + MILP: Candidate Arc Screening in a Supply Chain

This notebook uses a **Relational Graph Convolutional Network (R-GCN)** inside a realistic hybrid optimization pattern.

The supply chain has four stages:

```text
Supplier --supplies--> Plant --feeds--> Warehouse --ships_to--> Customer
```

The full model is a fixed-charge network-design MILP. Each candidate arc has:

- a continuous flow variable,
- a binary activation variable,
- variable shipping cost,
- fixed activation cost,
- capacity.

The pipeline is:

```text
supply-chain instance
        ↓
full MILP
        ↓
arcs activated in the optimum = training labels
        ↓
R-GCN edge scorer
        ↓
candidate arc screening
        ↓
reduced MILP
        ↓
feasibility + objective gap + retained arcs + solve time
```

The GNN does **not** replace the optimizer. It proposes a smaller candidate network, and the MILP solver validates the final decision.


In [ ]:
import random
import time
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

from scipy.optimize import milp, LinearConstraint, Bounds
from torch_geometric.nn import RGCNConv

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


## 1. Generate multi-echelon instances

We use fixed node counts so that the example stays compact, but costs, capacities, supply, demand, and arc availability vary across instances.


In [ ]:
STAGES = {
    "supplier": range(0, 3),
    "plant": range(3, 6),
    "warehouse": range(6, 9),
    "customer": range(9, 13),
}
N_NODES = 13

REL = {
    "supplies": 0,
    "feeds": 1,
    "ships_to": 2,
    "rev_supplies": 3,
    "rev_feeds": 4,
    "rev_ships_to": 5,
}
N_REL = len(REL)


def _candidate_pairs(left, right):
    return [(i, j) for i in left for j in right]


BASE_PAIRS = (
    [(u, v, "supplies") for u, v in _candidate_pairs(STAGES["supplier"], STAGES["plant"])]
    + [(u, v, "feeds") for u, v in _candidate_pairs(STAGES["plant"], STAGES["warehouse"])]
    + [(u, v, "ships_to") for u, v in _candidate_pairs(STAGES["warehouse"], STAGES["customer"])]
)


def generate_instance(seed=0, keep_prob=0.85):
    rng = np.random.default_rng(seed)

    demand = rng.integers(4, 10, size=4).astype(float)
    total_demand = demand.sum()

    supply = rng.uniform(0.45, 0.70, size=3)
    supply = supply / supply.sum() * total_demand * 1.20

    arcs = []
    for u, v, relation in BASE_PAIRS:
        if rng.random() <= keep_prob:
            variable_cost = float(rng.uniform(0.8, 6.0))
            fixed_cost = float(rng.uniform(0.5, 5.0))
            capacity = float(rng.uniform(0.35, 0.80) * total_demand)
            arcs.append((u, v, relation, variable_cost, fixed_cost, capacity))

    # Guarantee basic stage connectivity.
    for c in STAGES["customer"]:
        if not any(v == c for _, v, *_ in arcs):
            w = int(rng.choice(list(STAGES["warehouse"])))
            arcs.append((w, c, "ships_to", float(rng.uniform(1, 5)), 2.0, float(total_demand)))

    for w in STAGES["warehouse"]:
        if not any(v == w and rel == "feeds" for _, v, rel, *_ in arcs):
            p = int(rng.choice(list(STAGES["plant"])))
            arcs.append((p, w, "feeds", float(rng.uniform(1, 5)), 2.0, float(total_demand)))

    for p in STAGES["plant"]:
        if not any(v == p and rel == "supplies" for _, v, rel, *_ in arcs):
            s = int(rng.choice(list(STAGES["supplier"])))
            arcs.append((s, p, "supplies", float(rng.uniform(1, 5)), 2.0, float(total_demand)))

    return {
        "supply": supply,
        "demand": demand,
        "arcs": arcs,
        "total_demand": float(total_demand),
    }


## 2. Full fixed-charge network-design MILP

For each candidate arc \(e\):

- \(f_e \ge 0\): flow,
- \(y_e\in\{0,1\}\): activation.

Capacity coupling:

\[
f_e \le U_e y_e.
\]

Plant and warehouse nodes use flow conservation. Customers receive exactly their demand, while suppliers cannot exceed available supply.


In [ ]:
def solve_network(inst, allowed=None, time_limit=20.0):
    arcs = inst["arcs"]
    m = len(arcs)

    if allowed is None:
        allowed = np.arange(m, dtype=int)
    else:
        allowed = np.array(sorted(set(map(int, allowed))), dtype=int)

    if len(allowed) == 0:
        return None

    selected_arcs = [arcs[i] for i in allowed]
    k = len(selected_arcs)

    variable_cost = np.array([a[3] for a in selected_arcs], dtype=float)
    fixed_cost = np.array([a[4] for a in selected_arcs], dtype=float)
    capacity = np.array([a[5] for a in selected_arcs], dtype=float)

    # Variables: [flow_0..flow_k-1, y_0..y_k-1]
    c = np.r_[variable_cost, fixed_cost]
    integrality = np.r_[np.zeros(k), np.ones(k)]
    lb = np.zeros(2 * k)
    ub = np.r_[capacity, np.ones(k)]

    rows = []
    lower = []
    upper = []

    # Arc capacity coupling: flow - cap*y <= 0
    for j in range(k):
        row = np.zeros(2 * k)
        row[j] = 1.0
        row[k + j] = -capacity[j]
        rows.append(row)
        lower.append(-np.inf)
        upper.append(0.0)

    # Supplier capacity.
    for local_s, node in enumerate(STAGES["supplier"]):
        row = np.zeros(2 * k)
        for j, (u, v, *_rest) in enumerate(selected_arcs):
            if u == node:
                row[j] += 1.0
        rows.append(row)
        lower.append(-np.inf)
        upper.append(float(inst["supply"][local_s]))

    # Plant and warehouse flow conservation.
    for node in list(STAGES["plant"]) + list(STAGES["warehouse"]):
        row = np.zeros(2 * k)
        for j, (u, v, *_rest) in enumerate(selected_arcs):
            if v == node:
                row[j] += 1.0
            if u == node:
                row[j] -= 1.0
        rows.append(row)
        lower.append(0.0)
        upper.append(0.0)

    # Customer demand equality.
    for local_c, node in enumerate(STAGES["customer"]):
        row = np.zeros(2 * k)
        for j, (u, v, *_rest) in enumerate(selected_arcs):
            if v == node:
                row[j] += 1.0
        d = float(inst["demand"][local_c])
        rows.append(row)
        lower.append(d)
        upper.append(d)

    A = np.vstack(rows)
    con = LinearConstraint(A, np.array(lower), np.array(upper))

    t0 = time.perf_counter()
    res = milp(
        c=c,
        integrality=integrality,
        bounds=Bounds(lb, ub),
        constraints=con,
        options={"time_limit": time_limit},
    )
    elapsed = time.perf_counter() - t0

    if not res.success or res.x is None:
        return None

    full_flow = np.zeros(m)
    full_y = np.zeros(m)
    full_flow[allowed] = res.x[:k]
    full_y[allowed] = np.rint(res.x[k:])

    return {
        "flow": full_flow,
        "y": full_y,
        "obj": float(np.dot(
            np.array([a[3] for a in arcs]), full_flow
        ) + np.dot(
            np.array([a[4] for a in arcs]), full_y
        )),
        "time": elapsed,
    }


example = generate_instance(SEED)
example_solution = solve_network(example)
print("Candidate arcs:", len(example["arcs"]))
print("Objective:", example_solution["obj"] if example_solution else None)


## 3. Build a multi-relational graph

Forward relations are:

```text
supplies, feeds, ships_to
```

and reverse relations are added so that information can propagate in both directions:

```text
rev_supplies, rev_feeds, rev_ships_to
```

The R-GCN therefore knows that a supplier-to-plant relation is not the same type of edge as a warehouse-to-customer relation.


In [ ]:
def node_features(inst):
    x = np.zeros((N_NODES, 6), dtype=np.float32)

    # One-hot stage type.
    for i in STAGES["supplier"]:
        x[i, 0] = 1.0
    for i in STAGES["plant"]:
        x[i, 1] = 1.0
    for i in STAGES["warehouse"]:
        x[i, 2] = 1.0
    for i in STAGES["customer"]:
        x[i, 3] = 1.0

    scale = max(inst["total_demand"], 1.0)

    for local, node in enumerate(STAGES["supplier"]):
        x[node, 4] = inst["supply"][local] / scale

    for local, node in enumerate(STAGES["customer"]):
        x[node, 5] = inst["demand"][local] / scale

    return x


def graph_tensors(inst):
    src = []
    dst = []
    edge_type = []

    forward_pairs = []
    edge_features = []

    max_var_cost = max(a[3] for a in inst["arcs"])
    max_fixed = max(a[4] for a in inst["arcs"])
    max_cap = max(a[5] for a in inst["arcs"])

    for idx, (u, v, rel, var_cost, fixed_cost, cap) in enumerate(inst["arcs"]):
        rid = REL[rel]
        rev = REL["rev_" + rel]

        src.extend([u, v])
        dst.extend([v, u])
        edge_type.extend([rid, rev])

        forward_pairs.append((u, v))
        edge_features.append([
            var_cost / max(max_var_cost, 1e-9),
            fixed_cost / max(max_fixed, 1e-9),
            cap / max(max_cap, 1e-9),
        ])

    return {
        "x": torch.tensor(node_features(inst), dtype=torch.float32),
        "edge_index": torch.tensor([src, dst], dtype=torch.long),
        "edge_type": torch.tensor(edge_type, dtype=torch.long),
        "forward_pairs": torch.tensor(forward_pairs, dtype=torch.long),
        "edge_features": torch.tensor(edge_features, dtype=torch.float32),
    }


## 4. R-GCN edge scorer

The node encoder uses relation-specific message passing. The edge scorer then combines source embedding, destination embedding, and arc attributes.


In [ ]:
class SupplyChainRGCN(nn.Module):
    def __init__(self, hidden=64):
        super().__init__()
        self.in_proj = nn.Linear(6, hidden)
        self.rgcn1 = RGCNConv(hidden, hidden, num_relations=N_REL)
        self.rgcn2 = RGCNConv(hidden, hidden, num_relations=N_REL)

        self.edge_head = nn.Sequential(
            nn.Linear(2 * hidden + 3, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1),
        )

    def forward(self, g):
        h = F.relu(self.in_proj(g["x"]))
        h = F.relu(self.rgcn1(h, g["edge_index"], g["edge_type"]))
        h = F.relu(self.rgcn2(h, g["edge_index"], g["edge_type"]))

        pairs = g["forward_pairs"]
        edge_h = torch.cat([
            h[pairs[:, 0]],
            h[pairs[:, 1]],
            g["edge_features"],
        ], dim=-1)

        return self.edge_head(edge_h).squeeze(-1)


def to_device(g):
    return {k: v.to(device) if torch.is_tensor(v) else v for k, v in g.items()}


## 5. Create labels from exact MILP solutions and train

Each optimal binary activation vector `y` becomes an edge-selection label.


In [ ]:
def build_dataset(n_instances, start_seed):
    data = []
    seed = start_seed

    while len(data) < n_instances:
        inst = generate_instance(seed)
        sol = solve_network(inst)
        seed += 1

        if sol is None:
            continue

        g = graph_tensors(inst)
        y = torch.tensor(sol["y"], dtype=torch.float32)
        data.append((inst, sol, g, y))

    return data


train_data = build_dataset(100, 1000)
test_data = build_dataset(30, 5000)

model = SupplyChainRGCN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=2e-3)

positives = sum(float(y.sum()) for *_rest, y in train_data)
total = sum(y.numel() for *_rest, y in train_data)
pos_weight = torch.tensor(
    max((total - positives) / max(positives, 1.0), 1.0),
    device=device,
)

for epoch in range(70):
    order = np.random.permutation(len(train_data))
    running = 0.0

    for idx in order:
        inst, sol, g, y = train_data[idx]
        g = to_device(g)
        y = y.to(device)

        logits = model(g)
        loss = F.binary_cross_entropy_with_logits(
            logits,
            y,
            pos_weight=pos_weight,
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running += float(loss)

    if (epoch + 1) % 20 == 0:
        print(f"epoch={epoch+1:3d} loss={running/len(train_data):.4f}")


## 6. Candidate arc screening with adaptive feasibility fallback

A fixed pruning fraction can accidentally remove necessary network structure. We therefore increase the retained fraction if the reduced MILP is infeasible.


In [ ]:
@torch.no_grad()
def predict_scores(model, g):
    model.eval()
    prob = torch.sigmoid(model(to_device(g)))
    return prob.cpu().numpy()


def cost_only_scores(inst):
    # Higher score = cheaper and more capacitated.
    arr = np.array([
        cap / max(var_cost + fixed_cost, 1e-9)
        for _u, _v, _rel, var_cost, fixed_cost, cap in inst["arcs"]
    ])
    return arr / max(arr.max(), 1e-9)


def solve_with_adaptive_pruning(inst, scores, start_fraction=0.45):
    m = len(scores)

    for fraction in [start_fraction, 0.60, 0.75, 0.90, 1.00]:
        k = max(1, int(np.ceil(m * fraction)))
        keep = np.argsort(scores)[-k:]
        sol = solve_network(inst, allowed=keep)
        if sol is not None:
            return sol, keep, fraction

    return None, np.arange(m), 1.0


## 7. Evaluate the optimization pipeline

The relevant metrics are not only classification accuracy. We measure:

- recall of arcs used by the full optimum,
- retained candidate ratio,
- feasibility,
- objective gap,
- full and reduced solve times.


In [ ]:
def evaluate(method="gnn"):
    rows = []

    for inst, full, g, y in test_data:
        if method == "gnn":
            scores = predict_scores(model, g)
        elif method == "cost":
            scores = cost_only_scores(inst)
        else:
            raise ValueError(method)

        reduced, keep, fraction = solve_with_adaptive_pruning(inst, scores)

        optimal_active = set(np.flatnonzero(full["y"] > 0.5).tolist())
        kept = set(map(int, keep))

        recall = len(optimal_active & kept) / max(len(optimal_active), 1)
        feasible = reduced is not None
        gap = np.nan if reduced is None else (
            100.0 * (reduced["obj"] - full["obj"]) / max(abs(full["obj"]), 1e-9)
        )

        rows.append({
            "feasible": feasible,
            "recall": recall,
            "retained": len(keep) / len(inst["arcs"]),
            "gap_pct": gap,
            "full_time": full["time"],
            "reduced_time": np.nan if reduced is None else reduced["time"],
        })

    return rows


def summarize(name, rows):
    arr = lambda k: np.array([r[k] for r in rows], dtype=float)

    return {
        "method": name,
        "feasibility": arr("feasible").mean(),
        "optimal_arc_recall": arr("recall").mean(),
        "retained_ratio": arr("retained").mean(),
        "mean_gap_pct": np.nanmean(arr("gap_pct")),
        "mean_full_time": arr("full_time").mean(),
        "mean_reduced_time": np.nanmean(arr("reduced_time")),
    }


results = [
    summarize("R-GCN", evaluate("gnn")),
    summarize("Cost-only heuristic", evaluate("cost")),
]
results


## 8. Interpretation

This is a genuine relational-graph use case because the graph contains semantically different edge types:

```text
supplier --supplies--> plant
plant --feeds--> warehouse
warehouse --ships_to--> customer
```

The useful production pattern is:

```text
R-GCN -> candidate arc scores -> conservative screening -> MILP solver -> validated decision
```

For serious benchmarking, add:

- larger instances,
- distribution-shift tests,
- stronger network-design heuristics,
- solver-native presolve and MIP-gap controls,
- multiple random seeds,
- end-to-end time measurements including GNN inference,
- calibration of pruning thresholds.
